The extracted csv files contain some unwanted text such as chapter titles, book names, and stray numbers. We will use regex patterns to identify and remove these unwanted texts from the 'content' column of our dataframe.

### 1. Remove the first line only (chapter title / book name + stray numbers)

In [ ]:
import pandas as pd

file_name = "HCIP_9_British_Paramountcy_And_Indian_Renaissance_1"

dfn = pd.read_csv(f"{file_name}.csv")

In [ ]:
def remove_first_line(text):
    if not isinstance(text, str):
        return text  # keep NaN as-is

    lines = text.splitlines()
    return "\n".join(lines[1:]).strip() if len(lines) > 1 else text


dfn['updated_content'] = dfn['content'].apply(remove_first_line)


In [ ]:
# def remove_last_line(text):
#     if not isinstance(text, str):
#         return text  # keep NaN as-is

#     lines = text.splitlines()
#     return "\n".join(lines[:-1]).strip() if len(lines) > 1 else text


# dfn['updated_content'] = dfn['content'].apply(remove_last_line)


Check in case we removed any important text. If the difference in word count is significant, we might need to check the content carefully.

In [ ]:
dfn["difference"] = dfn["content"].astype(str).str.split().str.len()- dfn["updated_content"].astype(str).str.split().str.len()

If everything looks good, update the columsn 'content' with the cleaned text.

In [ ]:
dfn['content'] = dfn['updated_content']
dfn = dfn.drop(columns=['updated_content', 'difference'])

In [ ]:
dfn.to_csv(f"data_csv/To_Do/{file_name}.csv", index=False, encoding="utf-8-sig")

### 2. Data Extraction & Fixing

The data we extracted has some inconsistencies in chapter numbers and titles. We will fix these inconsistencies by manually assigning the correct chapter numbers and titles.

In [ ]:
import re

def extract_chapter_no(chapter_str):
    match = re.search(r'Chapter\s+([IVXLCDM]+|\d+)', chapter_str)
    if not match:
        return None
    roman = match.group(1)
    return roman_to_int(roman) if roman.isalpha() else int(roman)

def roman_to_int(s):
    roman_map = {'I':1,'V':5,'X':10,'L':50,'C':100,'D':500,'M':1000}
    total = 0
    prev = 0
    for c in reversed(s):
        val = roman_map[c]
        total += val if val >= prev else -val
        prev = val
    return total

df2 = dfn.copy()


df2['Chapter_No'] = df2['chapter'].apply(extract_chapter_no)
df2['Book_No'] = None
df2['Book_Name'] = None


In [ ]:
# Assuming df2 is already defined in your notebook as per the code above
page_ranges = df2.groupby('title').agg({'page': ['min', 'max'], 'Chapter_No': 'first'}).rename(columns={'min': 'start_page', 'max': 'end_page'})
page_ranges.columns = ['start_page', 'end_page', 'chapter_no']
page_ranges = page_ranges.sort_values('start_page')
page_ranges


In [ ]:
df2.loc[(df2['page'] >= 41) & (df2['page'] <= 49), 'Chapter_No'] = 1
df2.loc[(df2['page'] >= 41) & (df2['page'] <= 49), 'title'] = "Succession Of Governors-General"

df2.loc[(df2['page'] >= 94) & (df2['page'] <= 132), 'Chapter_No'] = 4
df2.loc[(df2['page'] >= 94) & (df2['page'] <= 132), 'title'] = "The Annexations Of Dalhousie"


In [ ]:
BOOK_MAPPING = [
    (1, "Political History", 1, 14),
    (2, "The Mutiny And the Revolt of 1857-58", 15, 22),
    (3, "India Under the British Crown (1858-1905)", 23, 33),
    (4, "Economic History (1818 to 1905)", 34, 38),
]

df2['Book_No'] = None
df2['Book_Name'] = None

for book_no, book_name, ch_start, ch_end in BOOK_MAPPING:
    mask = df2['Chapter_No'].between(ch_start, ch_end)
    df2.loc[mask, 'Book_No'] = book_no
    df2.loc[mask, 'Book_Name'] = book_name


In [ ]:
df2 = (
    df2.loc[:, ['Book_No', 'Book_Name', 'Chapter_No', 'title', 'page', 'content']]
       .rename(columns={'title': 'Chapter_Name'} )
)

Let's remove unwanted rows

In [ ]:
df2 = df2[

    ~(df2['page'].isin([23, 110]))
    & ~(df2['page'].between(72, 75))

 ]

df_final = df2

In [ ]:
df_final.to_csv(f"data_csv/{file_name}.csv", index=False, encoding="utf-8-sig")

### 3. Chunking Strategy: Recursive

Take small data from previous page and from next page and concatenate them in current page data to maintain context while chunking, also do not lose page number.

Then we will apply  recursive chunking strategy using `RecursiveCharacterTextSplitter` from langchain to split the cleaned text into smaller chunks suitable for embedding generation.

In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
def add_page_overlap(df, overlap_chars=200):
    df = df.sort_values(
        ["Chapter_No", "page"],
        na_position="first"
    ).reset_index(drop=True)

    overlapped_contents = []

    for i, row in df.iterrows():
        prev_tail = ""
        next_head = ""

        # This Condition will help is maintaing the Chapter wise structure other wise every thing will be cluttered.
        # Example Page 50 is Chapter 1 and Page 51 is Chapter 2 they will be same concatenated into one they shouldn't be.
        # Chapter 1's content shouldn't be in combined with Chapter 2, as it is a new chapter. and vice versa.

        if i > 0 and df.loc[i - 1, "Chapter_No"] == row["Chapter_No"]:
            prev_tail = df.loc[i - 1, "content"][-overlap_chars:]

        if i < len(df) - 1 and df.loc[i + 1, "Chapter_No"] == row["Chapter_No"]:
            next_head = df.loc[i + 1, "content"][:overlap_chars]

        combined_text = (
            prev_tail.strip()
            + "\n\n"
            + row["content"].strip()
            + "\n\n"
            + next_head.strip()
        ).strip()

        overlapped_contents.append(combined_text)

    df["content_with_overlap"] = overlapped_contents
    return df


In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1600,      # characters
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", "? ", "! ", " "]
)


In [4]:
def safe_part(val):
    if pd.isna(val):
        return "NA"
    return str(int(val)) if isinstance(val, (int, float)) else str(val)


In [ ]:
def chunk_pages(df):
    chunked_rows = []

    SERIES_INFO = {
        "series_id": SERIES_ID,
        "series_title": SERIES_TITLE,
    }

    VOLUME_INFO = {
        "volume_no": VOLUME_NO,
        "volume_title": VOLUME_TITLE,
    }

    for _, row in df.iterrows():
        if not row["content_with_overlap"]:
            continue

        chunks = text_splitter.split_text(row["content_with_overlap"])

        for idx, chunk in enumerate(chunks):
            chunked_rows.append({

                "historian": HISTORIAN,
                "series": SERIES_INFO,
                "volume": VOLUME_INFO,

                "book_no": row.get("Book_No"),
                "book_title": row.get("Book_Name"),
                "chapter_no": row.get("Chapter_No"),
                "chapter_title": row.get("Chapter_Name"),

                "page": row["page"],

                "chunk_id": (
                                f"{SERIES_ID}"
                                f"_V{VOLUME_NO}"
                                f"_B{safe_part(row.get('Book_No'))}"
                                f"_C{safe_part(row.get('Chapter_No'))}"
                                f"_P{int(row['page'])}"
                                f"_{idx}"
                            ),

                "content": chunk
            })

    return pd.DataFrame(chunked_rows)


In [ ]:
SERIES_ID = "HCIP_RCM"
SERIES_TITLE = "History and Culture of the Indian People"

VOLUME_NO = 9                              # None
VOLUME_TITLE = "British Paramountcy and Indian Renaissance - Part 1"       # None

HISTORIAN = "RC Majumdar"

#### Call the functions to chunk the data

In [ ]:
# Load your extracted pages
df = df_final

df["content"] = (
    df["content"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# Add page overlap
df = add_page_overlap(df)

# Chunk pages
chunked_df = chunk_pages(df)

# Save Chunks
chunked_df.to_json(
    f"new_data_jsonl/{file_name}.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

print("Chunking complete.")
print(f"Total chunks created: {len(chunked_df)}")



Chunking complete.
Total chunks created: 2467
